In [ ]:
## this code in originally Ruth's

In [3]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
library(Seurat)
library(ggplot2)
library(sctransform)
#library(scater)
library(reticulate)
library(future)
#library('gPCA')
library('Biobase')
#library(pheatmap)
#library("ggfortify")
#library('qvalue')
library(gplots)
#library('DESeq2')
#library(VennDiagram)
library('hdf5r')
library(EnsDb.Hsapiens.v86)
#library(EnsDb.Mmusculus.v79)
library(BiocParallel)
#library(tictoc)
library(Seurat)
library(Signac)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(cowplot)
library(GenomeInfoDb)
library(stats)
library("Signac")
library(tibble)
library(broom)
library("tictoc")
library("parallel")
library(tidyr)
library("dplyr")


Attaching package: 'gplots'


The following object is masked from 'package:IRanges':

    space


The following object is masked from 'package:S4Vectors':

    space


The following object is masked from 'package:stats':

    lowess



Attaching package: 'cowplot'


The following object is masked from 'package:ggpubr':

    get_legend



Attaching package: 'tictoc'


The following object is masked from 'package:data.table':

    shift


The following object is masked from 'package:GenomicRanges':

    shift


The following object is masked from 'package:IRanges':

    shift



Attaching package: 'tidyr'


The following objects are masked from 'package:Matrix':

    expand, pack, unpack


The following object is masked from 'package:S4Vectors':

    expand




### take logistic regression results -- make a bed file of sig peaks 

In [6]:
cells <- scan('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/celltypes.txt', what="", sep="\n")
cells

[1] "Beta"               "Alpha"              "Acinar_4"          
 [4] "Activated_Stellate" "Endothelial"        "Quiescent_Stellate"
 [7] "Tcells"             "Macrophage"         "Acinar_5"          
[10] "Ductal"

In [8]:
dir.create('inputs')

In [14]:
### get pval sig CREs
disease_use <- c('earlyT1D','Aab','lateT1D')

for (CELL in cells){
        for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_%s/%s.LogReg.txt', CELL,disease_use[disease_num]))
        if (file.exists(file_name)) {
            print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])

            res <- subset(res, subset = res[pval_col] < 0.05)
            res <- subset(res, select=-c(std.error,estimate))
            res <- select(res, -contains(c("fdr_")))
            res$celltype <- CELL
            res <- separate(res, peak, into = c("chr", "start", "end"), sep = "_")
            res <- res[,colnames(res) %in% c("chr", "start", "end")]

            write.table(res, sprintf('inputs/%s_%s_pval05.bed',CELL,disease_use[disease_num]), sep = '\t', quote = FALSE, col.names = FALSE, row.names = FALSE)
        }  
    }
} 

celltype = c('Beta')
disease_use <- c('earlyT1D','Aab','lateT1D')

for (CELL in celltype){
        for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_betaPropCovar/%s.LogReg.txt',disease_use[disease_num]))
        if (file.exists(file_name)) {
            print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])

            res <- subset(res, subset = res[pval_col] < 0.05)
            res <- subset(res, select=-c(std.error,estimate))
            res <- select(res, -contains(c("fdr_")))
            res$celltype <- CELL
            res <- separate(res, peak, into = c("chr", "start", "end"), sep = "_")
            res <- res[,colnames(res) %in% c("chr", "start", "end")]

            write.table(res, sprintf('inputs/%s_%s_pval05.bed',CELL,disease_use[disease_num]), sep = '\t', quote = FALSE, col.names = FALSE, row.names = FALSE)
        }  
    }
}    



[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_Alpha/earlyT1D.LogReg.txt"
[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_Alpha/Aab.LogReg.txt"
[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_Alpha/lateT1D.LogReg.txt"
[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_differences/051524_LogReg/logReg_results_WestonCovar_min10_Acinar_4/earlyT1D.LogReg.txt"
[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/disease_diff

### sort bed files

```bash
#!/bin/bash

# Loop over all .bed files in the current directory
for file in *.bed; do
  # Construct the output file name by appending .sorted.bed
  sorted_file="${file%.bed}.sorted.bed"
  # Sort the file by the first and second columns and save the result
  sort -k1,1 -k2,2n "$file" > "$sorted_file"
  echo "Sorted $file and saved to $sorted_file"
done

```

### get background peaks aka all peaks called in cell type

In [17]:
dir.create('backgrounds')
### get background peaks

for (CELL in cells){
    file_name <- paste0(sprintf('/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/%s_peaks.bed',CELL))
    if (file.exists(file_name)) {
            print(file_name)
            file.copy(file_name, 'backgrounds/')
        }  
    }



Warning message in dir.create("backgrounds"):
"'backgrounds' already exists"


[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Beta_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Alpha_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Acinar_4_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Activated_Stellate_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Endothelial_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Quiescent_Stellate_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/Tcells_peaks.bed"
[1] "/nfs/lab/projects/nPOD/downstream_files/ATAC/final

## Run Finrich

In [18]:
getwd()

[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/Finrich/finemap"

```bash
# test single finrich 
finrich /nfs/lab/hmummey/multiomic_islet/intermediates/221102_variant_overlaps/GWAS_credible_sets/T1D_JChiou_99credset.bed /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/Finrich/finemap/inputs/Acinar_4_Aab_pval05.sorted.bed /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/Finrich/finemap/backgrounds/Acinar_4_peaks.bed --permutations 1000 --processes 2

```

In [20]:
trait_beds <- list.files('inputs/', full.names = TRUE) #full.names makes sure you have the full file path
cell_type_beds <- list.files('', full.names = TRUE)


background_beds <- list.files('backgrounds/', full.names = TRUE)

results2 <- NULL #Create an empty dataframe to store results from loop

for (CELL in cells){
        for (disease_num in 1:length(disease_use)){
            input_name <- paste0(sprintf('inputs/%s_%s_pval05.sorted.bed', CELL,disease_use[disease_num]))
            if (file.exists(input_name)) {
                background_name <- paste0(sprintf('backgrounds/%s_peaks.bed', CELL))
                res <- system(paste0('finrich /nfs/lab/hmummey/multiomic_islet/intermediates/221102_variant_overlaps/GWAS_credible_sets/T1D_JChiou_99credset.bed ', input_name, ' ', background_name,' --permutations 1000 --processes 2'), intern = TRUE)
                res <- strsplit(res, ":|,|}")
                res <- as.data.frame(res)
                if (length(res > 0)){
                    results <- data.frame(matrix(ncol=0, nrow=1))
                    results$disease_condition <- disease_use[disease_num]
                    results$cell_type <- CELL
                    results$pval <- res[2,]
                    results$fold_enrich <- res[4,]
                    results$logOR <- res[6,]
                    results$conf_lower <- res[8,]
                    results$conf_upper  <- res[10,]
                    results2 <- rbind(results, results2)
                }
        }
    }
}


results2$pval <- as.numeric(results2$pval)
results2$log10p <- -log10(results2$pval)
head(results2)

,disease_condition,cell_type,pval,fold_enrich,logOR,conf_lower,conf_upper,log10p
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,lateT1D,Ductal,0.265,1.2747026788854656,0.2527932672761093,-1.2108462794341315,4.33304042232603,0.5767541
2,Aab,Ductal,0.284,1.0747629461257127,0.07391675564461608,-1.5450924633012313,5.8735202234305275,0.5466817
3,earlyT1D,Ductal,0.623,0.3715477025967056,-1.0132410037971018,-2.529715229080508,3.436564246815251,0.2055120
4,lateT1D,Acinar_5,0.088,3.595239768746183,1.3836897440112912,-0.3636686834424716,7.962011498445181,1.0555173
5,Aab,Acinar_5,0.121,2.6241596084595895,1.0240381870776265,-0.7290695619151834,7.80621420439549,0.9172146
6,earlyT1D,Acinar_5,0.022,6.880413547603774,2.0730696689148997,0.05735570050588839,14.460783715195662,1.6575773


In [22]:
results2$fdr <- p.adjust(results2$pval, method = 'BH')
results2

disease_condition,cell_type,pval,fold_enrich,logOR,conf_lower,conf_upper,log10p,fdr
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
lateT1D,Ductal,0.265,1.2747026788854656,0.2527932672761093,-1.2108462794341315,4.33304042232603,0.57675413,0.5015294
Aab,Ductal,0.284,1.0747629461257127,0.07391675564461608,-1.5450924633012313,5.8735202234305275,0.54668166,0.5015294
earlyT1D,Ductal,0.623,0.3715477025967056,-1.0132410037971018,-2.529715229080508,3.436564246815251,0.20551195,0.7226800
lateT1D,Acinar_5,0.088,3.595239768746183,1.3836897440112912,-0.3636686834424716,7.962011498445181,1.05551733,0.4881667
Aab,Acinar_5,0.121,2.6241596084595895,1.0240381870776265,-0.7290695619151834,7.80621420439549,0.91721463,0.5012857
earlyT1D,Acinar_5,0.022,6.880413547603774,2.0730696689148997,0.05735570050588839,14.460783715195662,1.65757732,0.2126667
lateT1D,Macrophage,0.500,0.5718673333217735,-0.5702459350764282,-2.04471393193488,3.8169993027744233,0.30103000,0.6304348
Aab,Macrophage,0.230,1.539862841925114,0.4530076915387742,-0.8946695009626973,3.756240711690443,0.63827216,0.5015294
earlyT1D,Macrophage,0.101,1.985182081706244,0.7473137986322849,-0.42519541077694223,3.0307231216826906,0.99567863,0.4881667


In [23]:
write.table(results2,'results.txt', sep = '\t', quote = FALSE, col.names = TRUE, row.names = FALSE)

In [24]:
getwd()

[1] "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/Finrich/finemap"